# 07. Difficulty Routing — Cutting Frontier Token Cost

Frontier models like Sonnet are powerful but expensive ($3/$15 per MTok). Yet a large share of customer inquiries are **questions a small model answers perfectly well** — business hours, how to report a lost card. In this notebook the decision layer becomes a **traffic router**: we classify each question's difficulty with `/v1/systemone` and send **simple ones to Haiku 4.5 ($1/$5), complex ones to Sonnet 4.5 ($3/$15)**. Generation stays on Bedrock in both arms — only the router is a self-hosted decision model.

Relation to notebook 05 (Bedrock Intelligent Prompt Routing):

| | Bedrock IPR (notebook 05) | difficulty routing (this notebook) |
|---|---|---|
| picks between | Bedrock models of one family (Lite/Pro) | **two models of different cost tiers (Haiku vs Sonnet)** — no family constraint |
| router | managed (built-in predictor) | built by you (decision layer) |
| router cost | $1 / 1,000 requests | tens to hundreds of ms, ~zero marginal cost |
| control | one responseQualityDifference knob | you design the rubric, choices, fallback |

The two are complementary — use IPR when managed is enough; use the decision layer when you want to own the rubric and the fallback direction.


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys, urllib.request; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
print("decision server:", f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}")
print("health:", urllib.request.urlopen(f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}/health", timeout=5).status)
client, sonnet = W.make_clients()
haiku  = W.BedrockLLM(client, W.BEDROCK_MODEL_IDS["haiku-4-5"], W.PRICING["haiku-4-5"])
router = W.SystemOneDecision()

## 1. The router = a 2-choice decision

We call `/v1/systemone` with a difficulty rubric (`W.ROUTER_QUESTION`) and the choices `["complex", "simple"]`.

> **Fallback guardrail**: the order of the choices matters. If the router output is invalid, the decision backend falls back to `choices[0]` — and `complex` is first, so **errors always route to Sonnet (the high-quality side)**. Cost optimization must never become a quality incident.


In [ ]:
print("rubric:", W.ROUTER_QUESTION, "\n")
for q in (W.routing_questions()[0]["q"], W.routing_questions()[5]["q"]):
    d = router.decide(state_text=q, question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    print(f"[{d.label:<7s}] conf {d.confidence:.3f} {d.latency_ms:>6,.0f}ms | {q[:60]}")

## 2. The question set — 10 mixed-difficulty questions

5 simple FAQ questions and 5 that need multi-step reasoning or policy interpretation. `gold` is the intended label.


In [ ]:
from IPython.display import Markdown, display
rows = ["| # | question | gold |", "|---|---|---|"]
for i, item in enumerate(W.routing_questions(), 1):
    rows.append(f"| {i} | {item['q']} | {item['gold']} |")
display(Markdown("\n".join(rows)))

## 3. Baseline — all 10 questions to Sonnet 4.5

First, measure the token cost with no routing: everything goes to the frontier model. `BedrockLLM` computes `cost_usd` from the actual token counts of each call.


In [ ]:
baseline = []
for item in W.routing_questions():
    r = sonnet.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    baseline.append({"backend": "sonnet", "cost_usd": r.cost_usd,
                     "input_tokens": r.usage["input_tokens"],
                     "output_tokens": r.usage["output_tokens"],
                     "latency_ms": r.latency_ms})
    print(f"${r.cost_usd:.5f} {r.latency_ms:>7,.0f}ms | {item['q'][:55]}")
print(f'\ntotal: ${sum(b["cost_usd"] for b in baseline):.4f}')

## 4. Routed — simple to Haiku, complex to Sonnet

The same 10 questions, now through the router. Haiku calls are billed Bedrock tokens too, so we record the **real token cost of every call** — the savings come entirely from "did a cheaper model answer the same question".


In [ ]:
routed, correct = [], 0
for item in W.routing_questions():
    d = router.decide(state_text=item["q"], question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    correct += (d.label == item["gold"])
    target, name = (haiku, "haiku") if d.label == "simple" else (sonnet, "sonnet")
    r = target.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    routed.append({"backend": name, "cost_usd": r.cost_usd,
                   "input_tokens": r.usage["input_tokens"],
                   "output_tokens": r.usage["output_tokens"],
                   "latency_ms": r.latency_ms, "router_ms": d.latency_ms})
    print(f"route={d.label:<7s} (conf {d.confidence:.2f}, {d.latency_ms:>5,.0f}ms)"
          f" -> {name:<6s} ${r.cost_usd:.5f} | {item['q'][:45]}")
print(f'\nrouter accuracy vs gold: {correct}/10')

## 5. Cost comparison — honest accounting

We look at the token-cost savings and the monthly projection. The routing decisions themselves run on the GPU the decision layer already uses, so **the router adds zero cost here** — if you were to provision a GPU for routing alone, you would weigh that fixed cost against the savings (see the note under the table).


In [ ]:
report = W.routing_cost_report(baseline, routed, req_per_day=10_000)

### What to look at

- **Quality guardrail**: every complex question was answered by Sonnet, and the fallback on router error is `choices[0]=complex` (Sonnet) too. Savings come only from questions a small model handles well.
- **The router itself is nearly free**: tens to hundreds of ms per decision, no Bedrock call. Compare with IPR's $1 per 1,000 requests.
- **The price ratio caps the savings**: Haiku costs 1/3 of Sonnet, so with a 50% simple share the arithmetic ceiling is ~33%. The real number depends on the output-token distribution — compare with the measured value above.
- **It is sensitive to rubric wording**: the same lesson as notebook 05 — changing the simple/complex criteria changes the routing distribution. Validate accuracy on a sample of real traffic before production.
- The savings scale with the **share of simple traffic**. This lab uses a 5:5 mix; recompute with your real FAQ share (usually higher).


### Try it yourself

- Change `req_per_day` and watch the monthly savings move.
- 3-way routing: extend the choices to `["complex", "medium", "simple"]` and answer simple ones with a **canned FAQ reply and no model call at all** — how far does the savings rate go?
- Confidence-based routing: build a conservative policy that sends low-confidence `simple` verdicts to Sonnet anyway (possible precisely because the confidence is calibrated).
- Feed scenario utterances like `W.scenario(1)["question"]` to the router and see how they classify.
